# 4.23 — Biclustering / co-clustering
Biclustering / co-clustering turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise. Part 4 moves from prediction with labels to discovery without labels; here matrix indexing becomes the model because rows and columns are chosen together. Save a copy to Drive to edit.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import SpectralBiclustering
from sklearn.datasets import load_digits, load_iris, make_blobs, make_moons
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity, LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, StandardScaler


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def matrix_for_biclustering(X):
    X = np.asarray(X, dtype=float)
    return MinMaxScaler().fit_transform(X) + 0.001


def project_for_plot(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    centered = StandardScaler().fit_transform(X)
    u, s, vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ vt[:2].T


def preview_rungs(rungs):
    for name, X, y, k in rungs:
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(name, "shape=", X.shape, "k=", k, "labels=", counts)
        print(np.round(X[:3, : min(5, X.shape[1])], 3))


def make_anomaly_ladder():
    rungs = []

    X1 = np.array([[0.0, 0.0], [0.2, 0.0], [0.0, 0.1], [4.0, 4.0]])
    y1 = np.array([0, 0, 0, 1])
    rungs.append(("D1 hand normals + one outlier", X1, y1, 2))

    for idx, (name, X, labels, k) in enumerate(cluster_ladder()[1:], start=2):
        rng = np.random.default_rng(240 + idx)
        X = np.asarray(X, dtype=float)
        n_outliers = max(8, int(0.08 * len(X)))
        lo = X.min(axis=0)
        hi = X.max(axis=0)
        span = np.maximum(hi - lo, 1.0)
        low_outliers = lo - rng.uniform(1.5, 2.5, size=(n_outliers // 2, X.shape[1])) * span
        high_outliers = hi + rng.uniform(1.5, 2.5, size=(n_outliers - n_outliers // 2, X.shape[1])) * span
        outliers = np.vstack([low_outliers, high_outliers])
        X_aug = np.vstack([X, outliers])
        y_aug = np.concatenate([np.zeros(len(X), dtype=int), np.ones(n_outliers, dtype=int)])
        rungs.append((name.replace("clusters", "normal groups") + " + synthetic outliers", X_aug, y_aug, 2))

    return rungs


def make_transaction_ladder():
    ladders = []
    base = [
        {"bread", "milk"},
        {"bread", "milk", "eggs"},
        {"bread", "eggs"},
        {"milk", "eggs"},
    ]
    labels = np.array([0, 0, 1, 1])
    ladders.append(("D1 four baskets", base, labels, 2))

    for name, X, y, k in cluster_ladder()[1:]:
        X = StandardScaler().fit_transform(X)
        transactions = []
        for row in X:
            items = set()
            for j, value in enumerate(row[: min(8, X.shape[1])]):
                if value <= -0.5:
                    items.add(f"f{j}_low")
                elif value >= 0.5:
                    items.add(f"f{j}_high")
                else:
                    items.add(f"f{j}_mid")
            transactions.append(items)
        ladders.append((name + " discretized transactions", transactions, y, k))

    return ladders


def print_table(rows, metric_name):
    print(f"{'rung':<38} {metric_name:>12}")
    for name, value in rows:
        print(f"{name:<38} {value:>12.3f}")


## The concept, built once (D1): block mean
The lesson's residue is $$H(I,J)=\frac{1}{|I||J|}\sum_{i\in I,j\in J}(x_{ij}-\hat x_{ij})^2.$$ First we compute the block mean so the selected rows and columns are visible before the full residue.

In [ ]:

def block_mean(X, rows, cols):
    block = X[np.ix_(rows, cols)]
    return float(block.mean())

lesson_matrix = np.array([[5.0, 5.0], [5.0, 4.0], [1.0, 1.0], [1.0, 2.0]])
lesson_rows = [0, 1]
lesson_cols = [0, 1]
mean_value = block_mean(lesson_matrix, lesson_rows, lesson_cols)
print("lesson block mean", mean_value)
assert round(mean_value, 3) == 4.750


## The concept, built once (D1): row-plus-column residue
A bicluster is coherent when an additive row-plus-column prediction leaves small residuals. The reusable method below fits real row and column clusters with `SpectralBiclustering`, then scores each selected block with the same residue arithmetic.

In [ ]:

def block_residue(X, rows, cols):
    block = X[np.ix_(rows, cols)]
    grand = block.mean()
    row_effect = block.mean(axis=1, keepdims=True) - grand
    col_effect = block.mean(axis=0, keepdims=True) - grand
    prediction = grand + row_effect + col_effect
    residual = block - prediction
    return float(np.mean(residual ** 2))


def method(X, n_row_clusters, n_col_clusters):
    X_work = matrix_for_biclustering(X)
    model = SpectralBiclustering(n_clusters=(n_row_clusters, n_col_clusters), random_state=0)
    model.fit(X_work)
    residues = []
    for r in range(n_row_clusters):
        rows = np.where(model.row_labels_ == r)[0]
        for c in range(n_col_clusters):
            cols = np.where(model.column_labels_ == c)[0]
            if len(rows) > 0 and len(cols) > 0:
                residues.append(block_residue(X_work, rows, cols))
    return model.row_labels_, model.column_labels_, float(np.mean(residues))

lesson_matrix = np.array([[5.0, 5.0], [5.0, 4.0], [4.0, 5.0], [4.0, 4.0]])
residue_value = block_residue(lesson_matrix, [0, 1], [0, 1])
print("lesson additive residue", residue_value)
assert round(residue_value, 4) == 0.0625


## The dataset ladder
We adapt `cluster_ladder()` into sample-feature matrices. Hidden labels are kept only for after-the-fact ARI scoring; the biclustering algorithm sees only `X`.

In [ ]:

rungs = cluster_ladder()
preview_rungs(rungs)


## Run the same method across D1–D5
The same spectral biclustering call runs on every rung. The metric is row-cluster ARI against hidden generator labels where available.

In [ ]:

rows = []
artifacts = []
for name, X, y_true, k in rungs:
    n_col_clusters = min(3, X.shape[1])
    row_labels, col_labels, mean_residue = method(X, k, n_col_clusters)
    score = adjusted_rand_score(y_true, row_labels)
    rows.append((name, score))
    artifacts.append((name, X, y_true, row_labels, col_labels, mean_residue))
print_table(rows, "row ARI")


## Results visualization
The left panels show reordered matrices, one per rung, and the right curve tracks ARI from D1 to D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, artifact in zip(flat_axes, artifacts):
    name, X, y_true, row_labels, col_labels, mean_residue = artifact
    X_work = matrix_for_biclustering(X)
    row_order = np.argsort(row_labels)
    col_order = np.argsort(col_labels)
    ax.imshow(X_work[row_order][:, col_order], aspect="auto", cmap="viridis")
    ax.set_title(name.split("(")[0] + f"\nresidue={mean_residue:.3f}")
    ax.set_xlabel("columns reordered")
    ax.set_ylabel("rows reordered")
flat_axes[-1].plot(range(1, 6), [value for _, value in rows], marker="o")
flat_axes[-1].set_xticks(range(1, 6))
flat_axes[-1].set_xlabel("rung")
flat_axes[-1].set_ylabel("row ARI")
flat_axes[-1].set_title("ARI vs. complexity")
plt.tight_layout()


## Pitfall on D5: tiny coherent blocks can overfit noise
A lower training residue can be a better fit to noise, not a better discovery. We create a random matrix with tiny blocks, then fix the pitfall by checking held-out columns and row-label stability.

In [ ]:

rng = np.random.default_rng(423)
_, X_d5, y_d5, k_d5 = rungs[-1]
X_noise = rng.normal(size=X_d5.shape)
train_cols, hold_cols = train_test_split(np.arange(X_noise.shape[1]), test_size=0.4, random_state=0)
labels_noise, col_noise, train_residue = method(X_noise[:, train_cols], k_d5, min(4, len(train_cols)))
hold_residue = block_residue(X_noise, np.where(labels_noise == labels_noise[0])[0], hold_cols[:8])
labels_a, _, _ = method(X_d5, k_d5, 4)
labels_b, _, _ = method(X_d5 + rng.normal(0.0, 0.02, size=X_d5.shape), k_d5, 4)
stability = adjusted_rand_score(labels_a, labels_b)
print("noise training residue", round(train_residue, 4))
print("held-out residue check", round(hold_residue, 4))
print("D5 stability after tiny perturbation", round(stability, 3))


## Evaluate it + Practice
- Compare the rung metric with a no-skill baseline: random row labels, random anomaly scores, one global density, or independence-only rules.
- Run a cheap sanity check: shuffle one key input and confirm the metric or discovered artifact degrades.
- Ablate the key idea: remove scaling, held-out scoring, bandwidth selection, or lift filtering and watch the metric drop.
- Inspect failure signals: unstable assignments, high training-only fit, score thresholds that flag whole subgroups, or rules with high support but lift near 1.
- Keep the hidden labels for evaluation only; the unsupervised method must not see them during fitting.

Practice prompts:
1. Change one hyperparameter near the default and predict which rung is most sensitive.


In [ ]:
# Your experiment for practice prompt 1 goes here.

2. Add a scaling or stability check and describe the before/after metric.

In [ ]:
# Your experiment for practice prompt 2 goes here.

3. Replace the D1 toy data with your own four-to-six point example and keep the asserts auditable.

In [ ]:
# Your experiment for practice prompt 3 goes here.